Scarping free patterns from https://www.crochet.com/patterns/free

In [97]:
import requests
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

In [98]:
def scrape_product(product_link):
    scraper = cloudscraper.create_scraper()

    r = scraper.get(product_link)
    soup = BeautifulSoup(r.text, "html.parser")

    row = {}

    button = None
    for b in soup.find_all("button"):
        if "Pattern Description" in b.get_text():
            button = b
            desc_div = button.find_parent("h3").find_next_sibling("div")
            row["description"] = desc_div.get_text(" ", strip=True)
        
        if "Pattern Details" in b.get_text():
            button_details = b
            details_div = button_details.find_parent("h3").find_next_sibling("div")
            items = details_div.find_all("li")

            for li in items:
                label = li.find("b")

                if label:
                    key = label.get_text(strip=True).replace(":", "")
                    value = label.next_sibling.strip()

                    row[key] = value
        
    return row

In [99]:
url = "https://www.crochet.com/filet-crochet-market-bag/p/58066"

scrape_product(url)

{'description': "Make a stunning market bag for trips to the farmers' market, outings to the beach, or to carry a few project bags worth of crochet goodies. You'll want to take this customizable bag everywhere! Practice the filet crochet technique with this mesh bag. Create patterns in the shape of a heart, a flower, a chevron pattern, or create a chart all your own. You will be a filet crochet expert by the time you finish this fast and fun market bag. Grab a project bundle and save!",
 'Pattern Type': 'Crochet',
 'Difficulty': 'Intermediate',
 'Sizes Included': '27" circumference, 15" long',
 'Yarn Weight': 'DK',
 'Yarn Line': '',
 'Yardage': '436',
 'Fiber Type': 'Cotton',
 'Needle / Hook Sizes': 'Size G (4.0mm) Crochet hook'}

In [100]:
base_url = "https://www.crochet.com/patterns/free"
rows = []
page = 1

scraper = cloudscraper.create_scraper()

while True:
    print(f"Scraping page {page}...")

    if page == 1:
        url = base_url
    else:
        url = f"{base_url}?Free-Paid=Free&page={page}"

    if page == 8:
        break

    response = scraper.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("div.group.relative")
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        link = card.select_one("a[href]")
        product_link = "https://www.crochet.com" + link["href"] 

        img_tag = card.select_one("img")
        image_link = img_tag["src"] if img_tag else None

        name_tag = card.select_one("h3 a")
        name = name_tag.get_text(strip=True) if name_tag else None
        
        product_data = scrape_product(product_link)

        product_data["name"] = name
        product_data["pattern_link"] = product_link
        product_data["image_link"] = image_link

        rows.append(product_data)
        time.sleep(1)

    page += 1

print("Scraping completed. Total products scraped:", len(rows))
df = pd.DataFrame(rows)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...


In [101]:
df.head()

,description,Pattern Type,Difficulty,Sizes Included,Yarn Weight,Yarn Line,Yardage,Fiber Type,Needle / Hook Sizes,name,pattern_link,image_link
0,Save when you buy a project bundle! Striped or...,Crochet,Beginner,"Bright Colorway, Cabin Colorway, Candy Colorway",Worsted,,570,Cotton,Size G (4.0mm) crochet hook,Wavy Chevron Dishcloth,https://www.crochet.com/wavy-chevron-dishcloth...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
1,Make a stunning market bag for trips to the fa...,Crochet,Intermediate,"27"" circumference, 15"" long",DK,,436,Cotton,Size G (4.0mm) Crochet hook,Filet Crochet Market Bag,https://www.crochet.com/filet-crochet-market-b...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
2,Get cozy in color with a striped crochet blank...,Crochet,Easy,"35.75 x 48"" - spring picnic, 48.25 x 60"" - spr...",Worsted,,1744-2616,Acrylic,US 7 (4.5mm) crochet hook,Good Earth Blanket,https://www.crochet.com/good-earth-blanket/p/N...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
3,Bold and modern the geometric mosaic pattern o...,Crochet,Intermediate,"38.5"" finished chest, 42.75"" finished chest, 4...",Worsted,,2616-3924,Acrylic,H/5.0mm & I/5.5mm crochet hooks,Aberdeen Pullover,https://www.crochet.com/aberdeen-pullover/p/56291,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
4,Playful vehicles set off for adventure on this...,Crochet,Intermediate,"24"" Wide x 36"" Long",Fingering,,1308,Cotton,US D/3 (3.25mm) Crochet hook,Adventure Time Blanket,https://www.crochet.com/adventure-time-blanket...,https://d2q9kw5vp0we94.cloudfront.net/i/w=1000...
